In [ ]:
# Import Required Libraries
import os

DATASETS_ROOT = "dataSets"

if __name__ == "__main__":
    if not os.path.isdir(DATASETS_ROOT):
        print(f"Datasets root directory '{DATASETS_ROOT}' not found.")
    else:
        for dataset_name in os.listdir(DATASETS_ROOT):
            dataset_path = os.path.join(DATASETS_ROOT, dataset_name)
            if os.path.isdir(dataset_path):
                print(f"Processing dataset: {dataset_name}") #testing dataset name

Processing dataset: indianexpress
Processing dataset: universityherald
Processing dataset: kotiliesi
Processing dataset: ruoka_fi
Processing dataset: theguardian


In [14]:
# Import Required Libraries
import os
from bs4 import BeautifulSoup
from collections import Counter

DATASETS_ROOT = "dataSets"
TARGET_TAGS = ["title", "h1", "h2", "h3", "h4", "p", "a", "strong", "b", "em"]


def analyze_local_dataset(dataset_dir):
    tag_counts = Counter()
    total_keywords_found = 0

    # Recursively find .htm/.html files in all subdirectories
    html_files = []
    for root, dirs, files in os.walk(dataset_dir):
        for file in files:
            if file.endswith(".htm") or file.endswith(".html"):
                html_files.append(os.path.join(root, file))

    print(f"Analyzing {len(html_files)} files in {dataset_dir}...")

    for file_path in html_files:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            soup = BeautifulSoup(f.read(), "lxml")

        # 1. Extract Ground Truth from Meta Tags
        gt_keywords = []
        # Extract from <meta name="keywords"> and <meta name="news_keywords">
        meta_tags = soup.find_all("meta", attrs={"name": True})
        for meta in meta_tags:
            name = meta.get("name", "").lower()
            if name in ["keywords", "news_keywords"] and meta.get("content"):
                gt_keywords += [k.strip().lower() for k in meta["content"].split(",")]

        # Extract from <meta property="article:tag" content="...">
        property_tags = soup.find_all("meta", attrs={"property": True})
        for meta in property_tags:
            prop = meta.get("property", "").lower()
            if prop == "article:tag" and meta.get("content"):
                gt_keywords.append(meta["content"].strip().lower())

        # Remove duplicates
        gt_keywords = list(set(gt_keywords))
        if not gt_keywords:
            continue

        # 2. Analyze occurrences in HTML elements
        for tag_name in TARGET_TAGS:
            elements = soup.find_all(tag_name)
            for el in elements:
                text = el.get_text().lower()
                for word in gt_keywords:
                    # Simple check if the keyword appears in the tag text
                    if word in text:
                        tag_counts[tag_name] += 1
                        total_keywords_found += 1

    return tag_counts, total_keywords_found


if __name__ == "__main__":
    if not os.path.isdir(DATASETS_ROOT):
        print(f"Datasets root directory '{DATASETS_ROOT}' not found.")
    else:
        for dataset_name in os.listdir(DATASETS_ROOT):
            dataset_path = os.path.join(DATASETS_ROOT, dataset_name)
            if os.path.isdir(dataset_path):
                stats, total_found = analyze_local_dataset(dataset_path)
                print(f"stats: {stats}, total_keywords_found: {total_found}")

Analyzing 658 files in dataSets/indianexpress...
stats: Counter({'a': 43601, 'p': 7134, 'title': 1216, 'h4': 1185, 'h1': 990, 'h2': 936, 'strong': 589, 'em': 56, 'b': 34, 'h3': 2}), total_keywords_found: 55743
Analyzing 600 files in dataSets/universityherald...
stats: Counter({'a': 44524, 'p': 37606, 'h3': 12352, 'title': 4008, 'h1': 3434, 'h2': 1968, 'strong': 1512, 'em': 34}), total_keywords_found: 105438
Analyzing 420 files in dataSets/kotiliesi...
stats: Counter({'a': 7909, 'p': 4228, 'h4': 752, 'title': 590, 'h1': 580, 'h3': 290, 'h2': 260, 'strong': 150, 'em': 134}), total_keywords_found: 14893
Analyzing 400 files in dataSets/ruoka_fi...
stats: Counter({'a': 4588, 'p': 2766, 'title': 760, 'h1': 698, 'strong': 252, 'h3': 174, 'b': 8, 'em': 4, 'h4': 4, 'h2': 2}), total_keywords_found: 9256
Analyzing 841 files in dataSets/theguardian...
stats: Counter({'a': 18964, 'p': 11989, 'h2': 2194, 'title': 1698, 'h1': 942, 'strong': 190, 'em': 140, 'h3': 22}), total_keywords_found: 36139


In [ ]:
# Import Required Libraries
import os
from bs4 import BeautifulSoup
from collections import Counter

DATASETS_ROOT = "dataSets"
TARGET_TAGS = ["title", "h1", "h2", "h3", "h4", "p", "a", "strong", "b", "em"]


def analyze_local_dataset(dataset_dir):
    """Analyzes HTML files in the given dataset directory to count occurrences of ground truth keywords in specific HTML tags."""
    tag_counts = Counter()
    total_keywords_found = 0

    # Recursively find .htm/.html files in all subdirectories
    html_files = []
    for root, dirs, files in os.walk(dataset_dir):
        for file in files:
            if file.endswith(".htm") or file.endswith(".html"):
                html_files.append(os.path.join(root, file))

    print(f"Analyzing {len(html_files)} files in {dataset_dir}...")

    for file_path in html_files:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            soup = BeautifulSoup(f.read(), "lxml")

        # 1. Extract Ground Truth from Meta Tags
        gt_keywords = []
        # Extract from <meta name="keywords"> and <meta name="news_keywords">
        meta_tags = soup.find_all("meta", attrs={"name": True})
        for meta in meta_tags:
            name = meta.get("name", "").lower()
            if name in ["keywords", "news_keywords"] and meta.get("content"):
                gt_keywords += [k.strip().lower() for k in meta["content"].split(",")]

        # Extract from <meta property="article:tag" content="...">
        property_tags = soup.find_all("meta", attrs={"property": True})
        for meta in property_tags:
            prop = meta.get("property", "").lower()
            if prop == "article:tag" and meta.get("content"):
                gt_keywords.append(meta["content"].strip().lower())

        # Remove duplicates
        gt_keywords = list(set(gt_keywords))
        if not gt_keywords:
            continue

        # 2. Analyze occurrences in HTML elements
        for tag_name in TARGET_TAGS:
            elements = soup.find_all(tag_name)
            for el in elements:
                text = el.get_text().lower()
                for word in gt_keywords:
                    # Simple check if the keyword appears in the tag text
                    if word in text:
                        tag_counts[tag_name] += 1
                        total_keywords_found += 1

    return tag_counts, total_keywords_found


def generate_report(counts, total, dataset_name):
    """Produces the ranked list and importance scores for Project 1."""
    print(f"\n===== Report for dataset: {dataset_name} =====")
    print(
        f"{'HTML Tag':<15} | {'Occurrences':<12} | {'Percentage':<12} | {'Importance Score':<12}"
    )
    print("-" * 75)

    # Ranking HTML tags by importance
    sorted_tags = counts.most_common(10)
    if not sorted_tags:
        print("No keywords found. Check if meta tags exist in the HTML.")
        return

    max_occ = sorted_tags[0][1]

    for tag, occ in sorted_tags:
        percentage = (occ / total) * 100 if total > 0 else 0
        importance_score = round(occ / max_occ, 2)  # Normalized
        tag_str = f"<{tag}>" + " " * (15 - len(tag) - 2)
        print(
            f"{tag_str} | {occ:<12} | {percentage:>10.2f}% | {importance_score:>15.2f}"
        )


if __name__ == "__main__":
    if not os.path.isdir(DATASETS_ROOT):
        print(f"Datasets root directory '{DATASETS_ROOT}' not found.")
    else:
        for dataset_name in os.listdir(DATASETS_ROOT):
            dataset_path = os.path.join(DATASETS_ROOT, dataset_name)
            if os.path.isdir(dataset_path):
                stats, total_found = analyze_local_dataset(dataset_path)
                # print(f"stats: {stats}, total_keywords_found: {total_found}")
                generate_report(stats, total_found, dataset_name)

Analyzing 658 files in dataSets/indianexpress...

===== Report for dataset: indianexpress =====
HTML Tag        | Occurrences  | Percentage   | Importance Score
---------------------------------------------------------------------------
<a>             | 43601        |      78.22% |            1.00
<p>             | 7134         |      12.80% |            0.16
<title>         | 1216         |       2.18% |            0.03
<h4>            | 1185         |       2.13% |            0.03
<h1>            | 990          |       1.78% |            0.02
<h2>            | 936          |       1.68% |            0.02
<strong>        | 589          |       1.06% |            0.01
<em>            | 56           |       0.10% |            0.00
<b>             | 34           |       0.06% |            0.00
<h3>            | 2            |       0.00% |            0.00
Analyzing 600 files in dataSets/universityherald...

===== Report for dataset: universityherald =====
HTML Tag        | Occurrences  